In [ ]:
import pandas as pd
import joblib

model = joblib.load("pipeline_billets_logreg.joblib")
print("Modèle chargé : pipeline_billets_logreg.joblib")

In [ ]:
file_path = input("Chemin du fichier CSV à analyser : ")
df_new = pd.read_csv(file_path)
print(f"Fichier chargé : {df_new.shape[0]} lignes, {df_new.shape[1]} colonnes")

In [ ]:
required_cols = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']

#Vérification des colonnes
missing_cols = [c for c in required_cols if c not in df_new.columns]
if missing_cols:
    raise ValueError(f"Colonnes manquantes : {missing_cols}\nColonnes attendues : {required_cols}")

#Extraction des features
X_new = df_new[required_cols].copy()

#Conversion en numérique
for col in required_cols:
    X_new[col] = pd.to_numeric(X_new[col], errors="coerce")

#Diagnostic des NaN
print("\nValeurs manquantes par colonne :")
print(X_new.isna().sum())

df_new["was_missing"] = X_new.isna().any(axis=1) 
print(f"\nLignes avec au moins une valeur manquante : {df_new['was_missing'].sum()}")

In [ ]:
pred = model.predict(X_new)
proba_true = model.predict_proba(X_new)[:, 1]   # proba d'être True (= billet vrai)

df_new["is_genuine_pred"] = pred
df_new["proba_true"] = proba_true

In [ ]:
print("\nAperçu des résultats :")
display(df_new[required_cols + ["was_missing", "is_genuine_pred", "proba_true"]].head(10))

print("\nRépartition des prédictions :")
print(df_new["is_genuine_pred"].value_counts())